# WP3: Stability of the (artificial) Lagrange points

## Personal problem specification 

| Parameter | Value |
|-----------|-------|
| Sun/Earth Lagrange point | L2 |
| Sun/Mars Lagrange point | L2 |
| Shift in Lagrange points | 0.00100 |
| Radius Martian parking orbit | 244,444 km |
| Lightness number | 0.0300 |

## Library & constant imports

In [21]:
from constants import *

import numpy as np
import scipy 
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import pickle
import nbimporter
from IPython.display import display

from wp1 import dUdx_colinear, nablaU

# ── Font sizes for plots (adjustable) ──────────────────────────────────────────
FONTSIZE_SMALL = 16              # Contour labels, small text
FONTSIZE_TICKS = 15              # Tick labels
FONTSIZE_AXIS_LABELS = 30        # Axis point labels
FONTSIZE_XLABEL_YLABEL = 16      # Axis label text
FONTSIZE_LEGEND = 15             # Legend and annotations
FONTSIZE_TITLE = 36              # Main titles

Loading the values from the previous computations. First the Lagrange points from WP1:

In [22]:
with open('lagrange_points.pkl', 'rb') as f:
    lagrange_points = pickle.load(f)

L2_earth = lagrange_points['L2_earth']
L2_mars  = lagrange_points['L2_mars']

And then the solar-sail lightness parameters from WP2:

In [23]:
with open('beta_values.pkl', 'rb') as f:
    beta_values = pickle.load(f)

beta_earth = beta_values['beta_earth']
beta_mars = beta_values['beta_mars']

Conversion of the points into the artificial lagrance points by the shift specified.

In [24]:
L2_artificial_earth = L2_earth - shift_L2
L2_artificial_mars  = L2_mars  - shift_L2

## Task 3.1a 

Compute the stability of the Lagrange points through the eigenvalues of the Jacobian matrix (ommitting the out-of-plane motion). 

For the classic CR3BP, the Jacobian matrix is:

$$J = \begin{bmatrix}
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1 \\
-U_{xx} & -U_{xy} & 0 & 2 \\
-U_{xy} & -U_{yy} & -2 & 0
\end{bmatrix}$$

where:

$$U_{xx} = 1 - \frac{1-\mu}{r_1^3} - \frac{\mu}{r_2^3} + \frac{3(1-\mu)(x+\mu)^2}{r_1^5} + \frac{3\mu(x-(1-\mu))^2}{r_2^5}$$

$$U_{yy} = 1 - \frac{1-\mu}{r_1^3} - \frac{\mu}{r_2^3} + \frac{3(1-\mu)y^2}{r_1^5} + \frac{3\mu y^2}{r_2^5}$$

$$U_{xy} = U_{yx} = \frac{3(1-\mu)(x+\mu)\,y}{r_1^5} + \frac{3\mu(x-(1-\mu))\,y}{r_2^5}$$

In [25]:
def hessianU(mu, x, y=0):
    """
    Computes the second-order partial derivatives of the effective potential U at (x, y) for a given mu.
    Returns (Uxx, Uxy, Uyy).
    """
    r1 = np.sqrt((x + mu)**2 + y**2)
    r2 = np.sqrt((x - (1 - mu))**2 + y**2)

    Uxx = (1
           - (1 - mu) / r1**3
           - mu / r2**3
           + 3 * (1 - mu) * (x + mu)**2 / r1**5
           + 3 * mu * (x - (1 - mu))**2 / r2**5)

    Uyy = (1
           - (1 - mu) / r1**3
           - mu / r2**3
           + 3 * (1 - mu) * y**2 / r1**5
           + 3 * mu * y**2 / r2**5)

    Uxy = (3 * (1 - mu) * (x + mu) * y / r1**5
           + 3 * mu * (x - (1 - mu)) * y / r2**5)

    return Uxx, Uxy, Uyy


def jacobian(mu, x, y=0):
    """
    Returns the 4x4 Jacobian matrix J of the linearised CR3BP equations of motion evaluated at position (x, y).
    """
    Uxx, Uxy, Uyy = hessianU(mu, x, y)

    J = np.array([
        [  0,     0,    1,  0],
        [  0,     0,    0,  1],
        [-Uxx, -Uxy,    0,  2],
        [-Uxy, -Uyy,   -2,  0]
    ])

    return J

### Classical CR3BP

For the Sun-Earth system, that yields the following Jacobian:

In [26]:
J_earth = jacobian(mass_ratio_sun_earth, L2_earth, 0)
print("J_earth:\n", J_earth)

J_earth:
 [[ 0.          0.          1.          0.        ]
 [ 0.          0.          0.          1.        ]
 [-8.88086731 -0.          0.          2.        ]
 [-0.          2.94043366 -2.          0.        ]]


And the following eigenvalues:

In [27]:
eigenvalues_earth, _ = np.linalg.eig(J_earth)
print("Eigenvalues of J_earth:")

for val in eigenvalues_earth:
    real_part = val.real if abs(val.real) > 1e-10 else 0.0
    imag_part = val.imag if abs(val.imag) > 1e-10 else 0.0
    
    if imag_part == 0:
        print(real_part)
    elif real_part == 0:
        print(f"{imag_part}j")
    else:
        print(complex(real_part, imag_part))

Eigenvalues of J_earth:
3.478332514656781j
-3.478332514656781j
-1.4691369666999288
1.4691369666999294


For the Sun-Mars system, that yields the following Jacobian:

In [28]:
J_mars = jacobian(mass_ratio_sun_mars, L2_mars, 0)
print("J_mars:\n", J_mars)

J_mars:
 [[ 0.          0.          1.          0.        ]
 [ 0.          0.          0.          1.        ]
 [-8.94357518 -0.          0.          2.        ]
 [-0.          2.97178759 -2.          0.        ]]


And the following eigenvalues:

In [29]:
eigenvalues_mars, _ = np.linalg.eig(J_mars)
print("Eigenvalues of J_mars:")

for val in eigenvalues_mars:
    real_part = val.real if abs(val.real) > 1e-10 else 0.0
    imag_part = val.imag if abs(val.imag) > 1e-10 else 0.0
    
    if imag_part == 0:
        print(real_part)
    elif real_part == 0:
        print(f"{imag_part}j")
    else:
        print(complex(real_part, imag_part))

Eigenvalues of J_mars:
3.486816730481088j
-3.486816730481088j
-1.4785477072729452
1.4785477072729445


### Solar-sail augmented CR3BP

Assuming fixed $\mathbf{\hat{n}} \parallel x$, the pseudo-potential of the augmented CR3BP can be written as: 

$$U^* = U + V_\text{sail} = \frac{1}{2}(x^2+y^2) + \frac{(1-\mu)+\beta\mu_\odot}{r_1} + \frac{\mu}{r_2} \Rightarrow \nabla U^* = \nabla U + \mathbf{a}_\text{sail} $$

$$U^*_{xx} = U_{xx} + \frac{\partial^2 V_\text{sail}}{\partial x^2}\bigg|_{y=0} = U_{xx} - \frac{2\beta\mu_\odot}{r_1^3}$$

$$U^*_{xy} = U_{xy}, \qquad U^*_{yy} = U_{yy}$$

In [30]:
def jacobian_sail(mu, beta, x, y=0):
    """
    Returns the 4x4 Jacobian matrix J of the linearised augmented CR3BP equations of motion evaluated at position (x, y).
    """
    r1 = np.sqrt((x + mu)**2 + y**2)

    Uxx, Uxy, Uyy = hessianU(mu, x, y)

    Uxx += -2*beta*mu_sun / r1**3

    J = np.array([
        [  0,     0,    1,  0],
        [  0,     0,    0,  1],
        [-Uxx, -Uxy,    0,  2],
        [-Uxy, -Uyy,   -2,  0]
    ])

    return J

For the Sun-Earth solar-sail augmented system, that yields the following Jacobian:

In [31]:
J_earth = jacobian_sail(mass_ratio_sun_earth, beta_earth, L2_earth, 0)
print("J_earth:\n", J_earth)

J_earth:
 [[ 0.00000000e+00  0.00000000e+00  1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]
 [ 2.59617660e+09 -0.00000000e+00  0.00000000e+00  2.00000000e+00]
 [-0.00000000e+00  2.94043366e+00 -2.00000000e+00  0.00000000e+00]]


And the following eigenvalues:

In [32]:
eigenvalues_earth, _ = np.linalg.eig(J_earth)
print("Eigenvalues of J_earth:")

for val in eigenvalues_earth:
    real_part = val.real if abs(val.real) > 1e-10 else 0.0
    imag_part = val.imag if abs(val.imag) > 1e-10 else 0.0
    
    if imag_part == 0:
        print(real_part)
    elif real_part == 0:
        print(f"{imag_part}j")
    else:
        print(complex(real_part, imag_part))

Eigenvalues of J_earth:
-50952.68974431295
50952.68974431296
-1.7147692731457207
1.7147692731457198


For the Sun-Mars solar-sail augmented system, that yields the following Jacobian:

In [33]:
J_mars = jacobian_sail(mass_ratio_sun_mars, beta_mars, L2_mars, 0)
print("J_mars:\n", J_mars)

J_mars:
 [[ 0.00000000e+00  0.00000000e+00  1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]
 [ 3.04196340e+09 -0.00000000e+00  0.00000000e+00  2.00000000e+00]
 [-0.00000000e+00  2.97178759e+00 -2.00000000e+00  0.00000000e+00]]


And the following eigenvalues:

In [34]:
eigenvalues_mars, _ = np.linalg.eig(J_mars)
print("Eigenvalues of J_mars:")

for val in eigenvalues_mars:
    real_part = val.real if abs(val.real) > 1e-10 else 0.0
    imag_part = val.imag if abs(val.imag) > 1e-10 else 0.0
    
    if imag_part == 0:
        print(real_part)
    elif real_part == 0:
        print(f"{imag_part}j")
    else:
        print(complex(real_part, imag_part))

Eigenvalues of J_mars:
-55153.99708696664
55153.997086966636
-1.7238873493324491
1.7238873493324485
